In [1]:
# Cellule 0 — Imports
import json, os, time
import numpy as np
import pandas as pd
from collections import defaultdict
from sentence_transformers import SentenceTransformer

TRANS_DIR = r'D:\Memoire\mooccubeX\translated'
RESULTS_REC = r'D:\Memoire\LLMKGER\reco_results_final'

print("✅ Imports OK")

# Charger les métadonnées des cours (nom, domaine, description)
course_info = {}
with open(os.path.join(TRANS_DIR, 'course_translated.json'), 'r', encoding='utf-8') as f:
    for line in f:
        try:
            obj = json.loads(line)
            cid = obj.get('id', '')
            if cid:
                course_info[cid] = {
                    'name': obj.get('name', ''),
                    'field': obj.get('field', []),
                    'about': (obj.get('about', '') or ''),
                }
        except: continue
print(f"Cours chargés : {len(course_info):,}")

# Vérifier combien ont une description
with_about = sum(1 for c in course_info.values() if c['about'].strip())
print(f"Cours avec description : {with_about:,}")

✅ Imports OK
Cours chargés : 3,781
Cours avec description : 3,692


In [2]:
# Charger le modèle d'embedding textuel
print("Chargement du modèle d'embedding...")
model_txt = SentenceTransformer('all-MiniLM-L6-v2')

# Ne garder que les cours candidats (ceux du système de reco)
# On a besoin de la liste des cours candidats
# Les charger depuis un fichier de référence
df_ref = pd.read_csv(os.path.join(RESULTS_REC, 'S_emb_scores_final.csv'))
# Les cours candidats sont ceux qui apparaissent comme recommandables
# On prend tous les cours du course_info qui sont dans le système

# Construire le texte représentatif de chaque cours
def course_text(cid):
    info = course_info.get(cid, {})
    name = info.get('name', '')
    fields = ', '.join(info.get('field', []))
    about = info.get('about', '')[:300]  # limiter
    return f"{name}. Field: {fields}. {about}".strip()

# Encoder tous les cours disponibles
course_ids = list(course_info.keys())
course_texts = [course_text(c) for c in course_ids]

print(f"Encodage de {len(course_ids):,} cours...")
t0 = time.time()
course_embeddings = model_txt.encode(course_texts, batch_size=128,
                                      show_progress_bar=True, convert_to_numpy=True,
                                      normalize_embeddings=True)
print(f"✅ Encodage terminé en {(time.time()-t0)/60:.1f} min")
print(f"   Shape : {course_embeddings.shape}")

# Index pour retrouver les cours
course_id_to_idx = {c: i for i, c in enumerate(course_ids)}
print(f"   {len(course_id_to_idx):,} cours indexés")

# Test : encoder un intérêt et voir les cours proches
test_interest = "I want to learn about electrical circuits and engineering"
test_emb = model_txt.encode([test_interest], normalize_embeddings=True)[0]
sims = course_embeddings @ test_emb
top5 = np.argsort(-sims)[:5]
print(f"\n=== Test : '{test_interest}' ===")
for idx in top5:
    print(f"  {sims[idx]:.3f} — {course_info[course_ids[idx]]['name']}")

Chargement du modèle d'embedding...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Encodage de 3,781 cours...


Batches:   0%|          | 0/30 [00:00<?, ?it/s]

✅ Encodage terminé en 0.1 min
   Shape : (3781, 384)
   3,781 cours indexés

=== Test : 'I want to learn about electrical circuits and engineering' ===
  0.739 — Circuit basics and applications
  0.690 — Principles of Electric Circuits
  0.682 — Electrical Fundamentals and Electronic Design
  0.676 — Circuit principle
  0.664 — Application and practice of electrical and electronic technology


In [3]:
from groq import Groq

# Remets ta clé Groq
GROQ_API_KEY = "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"
client = Groq(api_key=GROQ_API_KEY)

def cold_start_recommend(user_interest, top_k=5, use_llm_enrich=True):
    """Recommande des cours pour un nouveau user à partir de ses intérêts (zero-shot)."""

    # 1. (Optionnel) Enrichir les intérêts avec le LLM
    enriched = user_interest
    if use_llm_enrich:
        try:
            resp = client.chat.completions.create(
                model="llama-3.3-70b-versatile",
                messages=[{"role":"user","content":
                    f"A learner says: '{user_interest}'. List 5-8 specific academic topics or concepts they would want to study, comma-separated, no explanation."}],
                temperature=0.5, max_tokens=100,
            )
            enriched = user_interest + ". Related topics: " + resp.choices[0].message.content.strip()
        except:
            enriched = user_interest

    # 2. Encoder les intérêts (enrichis) et matcher
    interest_emb = model_txt.encode([enriched], normalize_embeddings=True)[0]
    sims = course_embeddings @ interest_emb
    top_idx = np.argsort(-sims)[:top_k]

    recommendations = []
    for idx in top_idx:
        cid = course_ids[idx]
        recommendations.append({
            'course_id': cid,
            'name': course_info[cid]['name'],
            'field': course_info[cid]['field'],
            'similarity': float(sims[idx])
        })
    return recommendations, enriched

# Test complet
interest = "I'm interested in machine learning and data analysis"
recs, enriched = cold_start_recommend(interest, top_k=5)

print(f"=== NOUVEAU USER : '{interest}' ===")
print(f"\n🤖 Intérêts enrichis par LLM :")
print(f"   {enriched}")
print(f"\n📚 RECOMMANDATIONS (zero-shot, sans historique) :")
for i, r in enumerate(recs, 1):
    print(f"  {i}. {r['name']} (sim={r['similarity']:.3f})")
    print(f"     Domaines : {', '.join(r['field']) if r['field'] else 'N/A'}")

=== NOUVEAU USER : 'I'm interested in machine learning and data analysis' ===

🤖 Intérêts enrichis par LLM :
   I'm interested in machine learning and data analysis. Related topics: Supervised learning, unsupervised learning, neural networks, deep learning, natural language processing, regression analysis, time series forecasting, clustering algorithms.

📚 RECOMMANDATIONS (zero-shot, sans historique) :
  1. machine learning (sim=0.699)
     Domaines : N/A
  2. data mining (sim=0.576)
     Domaines : N/A
  3. Introduction to Artificial Intelligence (sim=0.574)
     Domaines : N/A
  4. Mathematical models and optimization techniques (sim=0.558)
     Domaines : N/A
  5. Data Mining: Theory and Algorithms (sim=0.558)
     Domaines : Computer Science and Technology


In [4]:
import pandas as pd, os, json
from collections import defaultdict
import numpy as np

RESULTS_REC = r'D:\Memoire\LLMKGER\reco_results_final'

# Charger les cours test réels (vérité terrain)
df_test = pd.read_csv(os.path.join(RESULTS_REC, 'S_emb_scores_final.csv'))
print(f"Users avec cours test : {len(df_test):,}")
print(f"Colonnes : {df_test.columns.tolist()}")
print(df_test.head(3))

# On a besoin de l'historique de chaque user pour construire ses "intérêts"
# Vérifier si user_history_courses est en mémoire
try:
    print(f"\nuser_history_courses en mémoire : {len(user_history_courses):,} users")
    have_history = True
except NameError:
    print("\n⚠️ user_history_courses absent — à recharger depuis RecBole")
    have_history = False

Users avec cours test : 53,897
Colonnes : ['user_id', 'course_id', 'S_emb']
     user_id course_id     S_emb
0   U_100090  C_676642  5.466541
1  U_1000902  C_854866  6.889208
2  U_1001872  C_697270  5.094931

⚠️ user_history_courses absent — à recharger depuis RecBole


In [5]:
import gc
from recbole.config import Config
from recbole.data import create_dataset
from recbole.utils import init_seed, init_logger

RECBOLE_ROOT = r'D:\Memoire\LLMKGER\recbole_data_large'
base_config = {
    'data_path': RECBOLE_ROOT, 'dataset': 'llmkger',
    'USER_ID_FIELD':'user_id','ITEM_ID_FIELD':'item_id','RATING_FIELD':'rating',
    'load_col':{'inter':['user_id','item_id','rating']},
    'eval_args':{'split':{'LS':'valid_and_test'},'order':'RO','group_by':'user','mode':'full'},
    'metrics':['NDCG'],'topk':[10],'valid_metric':'NDCG@10',
    'epochs':1,'device':'cpu','seed':42,'show_progress':False,
}
config = Config(model='LightGCN', dataset='llmkger', config_dict=base_config)
init_seed(config['seed'], config['reproducibility']); init_logger(config)
dataset = create_dataset(config)

n_users = dataset.num(config['USER_ID_FIELD'])
n_items = dataset.num(config['ITEM_ID_FIELD'])
item2id = dict(dataset.field2token_id[config['ITEM_ID_FIELD']])
user2id = dict(dataset.field2token_id[config['USER_ID_FIELD']])
id2item = {v:k for k,v in item2id.items()}
id2user = {v:k for k,v in user2id.items()}
candidate_items = [i for i in range(n_items) if id2item.get(i,'[PAD]')!='[PAD]']
candidate_courses = [id2item[i] for i in candidate_items]

# Historique de chaque user
inter_all = dataset.inter_feat
uu = inter_all[config['USER_ID_FIELD']].to_numpy()
ii = inter_all[config['ITEM_ID_FIELD']].to_numpy()
user_history_courses = defaultdict(list)
for u,i in zip(uu,ii):
    u_ext = id2user.get(int(u)); c_ext = id2item.get(int(i))
    if u_ext and c_ext:
        user_history_courses[u_ext].append(c_ext)

print(f"✅ Historique : {len(user_history_courses):,} users")
print(f"   Cours candidats : {len(candidate_courses):,}")
del dataset; gc.collect()

c:\Users\AMIRA\anaconda3\envs\llmkger\lib\site-packages\recbole\data\dataset\dataset.py:648: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  feat[field].fillna(value=0, inplace=True)
c:\Users\AMIRA\anaconda3\envs\llmkger\lib\site-packages\recbole\data\dataset\dataset.py:650: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are sett

✅ Historique : 53,897 users
   Cours candidats : 2,812


29

In [7]:
import numpy as np, time
from collections import Counter

def user_interest_text(user_courses):
    fields = []
    for c in user_courses:
        fields.extend(course_info.get(c, {}).get('field', []))
    top_fields = [f for f,_ in Counter(fields).most_common(3)]
    if top_fields:
        return "I am interested in " + ", ".join(top_fields)
    return None

# Préparer tous les users de l'échantillon d'un coup
np.random.seed(42)
sample = np.random.choice(df_test['user_id'].tolist(), 500, replace=False)

# Construire les intérêts + cibles pour les users valides
eval_users = []
eval_interests = []
eval_targets = []
for u in sample:
    test_row = df_test[df_test['user_id']==u]
    if len(test_row)==0: continue
    target = test_row['course_id'].iloc[0]
    if target not in cand_valid_index: continue
    history = user_history_courses.get(u, [])
    interest = user_interest_text(history)
    if not interest: continue
    eval_users.append(u)
    eval_interests.append(interest)
    eval_targets.append(target)

print(f"Users à évaluer : {len(eval_users)}")

# Encoder TOUS les intérêts en UNE fois (batch, pas de barre)
print("Encodage des intérêts (batch)...")
interest_embs = model_txt.encode(eval_interests, batch_size=128,
                                  show_progress_bar=False, normalize_embeddings=True)

# Calculer les rangs
print("Calcul des scores...")
ndcg_cold=[]; recall_cold=[]
for i, target in enumerate(eval_targets):
    sims = candidate_embeddings @ interest_embs[i]
    target_pos = cand_valid_index[target]
    rank = int((sims > sims[target_pos]).sum()) + 1
    if rank <= 10:
        ndcg_cold.append(1.0/np.log2(rank+1)); recall_cold.append(1.0)
    else:
        ndcg_cold.append(0.0); recall_cold.append(0.0)

print(f"\n=== COLD-START (matching textuel, sans historique) ===")
print(f"  NDCG@10 : {np.mean(ndcg_cold):.4f}")
print(f"  Recall@10 : {np.mean(recall_cold):.4f}")
print(f"  Users évalués : {len(ndcg_cold)}")
print(f"\n  Référence hasard (1/2812) : {1/2812:.4f}")
print(f"  Référence warm (système complet) : 0.1136")

Users à évaluer : 500
Encodage des intérêts (batch)...
Calcul des scores...

=== COLD-START (matching textuel, sans historique) ===
  NDCG@10 : 0.0089
  Recall@10 : 0.0180
  Users évalués : 500

  Référence hasard (1/2812) : 0.0004
  Référence warm (système complet) : 0.1136


In [8]:
import numpy as np, time
from collections import Counter

# Few-shot : utiliser 1 cours de départ (≠ cours test) pour représenter les intérêts
def course_text(cid):
    info = course_info.get(cid, {})
    name = info.get('name', '')
    fields = ', '.join(info.get('field', []))
    about = info.get('about', '')[:300]
    return f"{name}. Field: {fields}. {about}".strip()

# Préparer l'évaluation few-shot
np.random.seed(42)
sample = np.random.choice(df_test['user_id'].tolist(), 500, replace=False)

fewshot_texts = []
fewshot_targets = []
for u in sample:
    test_row = df_test[df_test['user_id']==u]
    if len(test_row)==0: continue
    target = test_row['course_id'].iloc[0]
    if target not in cand_valid_index: continue

    history = user_history_courses.get(u, [])
    # Prendre un cours de départ DIFFÉRENT du cours test
    start_courses = [c for c in history if c != target]
    if not start_courses: continue

    # Utiliser le 1er cours de départ comme représentation des intérêts
    start_course = start_courses[0]
    fewshot_texts.append(course_text(start_course))
    fewshot_targets.append(target)

print(f"Users évaluables (few-shot) : {len(fewshot_texts)}")

# Encoder les cours de départ (batch)
print("Encodage (batch)...")
start_embs = model_txt.encode(fewshot_texts, batch_size=128,
                              show_progress_bar=False, normalize_embeddings=True)

# Calculer les rangs
ndcg_fs=[]; recall_fs=[]
for i, target in enumerate(fewshot_targets):
    sims = candidate_embeddings @ start_embs[i]
    target_pos = cand_valid_index[target]
    rank = int((sims > sims[target_pos]).sum()) + 1
    if rank <= 10:
        ndcg_fs.append(1.0/np.log2(rank+1)); recall_fs.append(1.0)
    else:
        ndcg_fs.append(0.0); recall_fs.append(0.0)

print(f"\n=== COLD-START FEW-SHOT (1 cours de départ) ===")
print(f"  NDCG@10 : {np.mean(ndcg_fs):.4f}")
print(f"  Recall@10 : {np.mean(recall_fs):.4f}")
print(f"  Users évalués : {len(ndcg_fs)}")
print(f"\n  Comparaison :")
print(f"    Hasard : 0.0004")
print(f"    Cold-start (domaines seuls) : 0.0089")
print(f"    Cold-start few-shot (1 cours) : {np.mean(ndcg_fs):.4f}")
print(f"    Warm (système complet) : 0.1136")

Users évaluables (few-shot) : 500
Encodage (batch)...

=== COLD-START FEW-SHOT (1 cours de départ) ===
  NDCG@10 : 0.0078
  Recall@10 : 0.0180
  Users évalués : 500

  Comparaison :
    Hasard : 0.0004
    Cold-start (domaines seuls) : 0.0089
    Cold-start few-shot (1 cours) : 0.0078
    Warm (système complet) : 0.1136


In [10]:
import pandas as pd, os
from collections import defaultdict

KG_DIR = r'D:\Memoire\LLMKGER\recbole_data_large\llmkger'
PREREQ_DIR = r'D:\Memoire\LLMKGER\prereq_enriched'

print("Reconstruction des structures KG...")

# Concepts d'un cours
kg = pd.read_csv(os.path.join(KG_DIR, 'llmkger.kg'), sep='\t')
kg.columns = ['head','relation','tail']
concepts_of_course = defaultdict(set)
for h,t in kg[kg['relation']=='belongs_to'][['head','tail']].itertuples(index=False):
    h,t=str(h),str(t)
    if h.startswith('K') and t.startswith('C_'): concepts_of_course[t].add(h)
    elif t.startswith('K') and h.startswith('C_'): concepts_of_course[h].add(t)
print(f"  Cours avec concepts : {len(concepts_of_course):,}")

# Prérequis entre concepts (enrichis)
prereq_df = pd.read_csv(os.path.join(PREREQ_DIR, 'prerequisite_of_enriched.csv'), sep='\t')
prereqs_of_concept = defaultdict(set)  # concept -> ses prérequis
concept_leads_to = defaultdict(set)    # concept -> concepts dont il est prérequis (progression)
for h, r, t in prereq_df.itertuples(index=False):
    h, t = str(h), str(t)
    prereqs_of_concept[t].add(h)     # h est prérequis de t
    concept_leads_to[h].add(t)       # h mène à t (progression)
print(f"  Concepts avec prérequis : {len(prereqs_of_concept):,}")
print(f"  Concepts menant à d'autres : {len(concept_leads_to):,}")

# Pour chaque cours, les concepts "avancés" (ce vers quoi ses concepts mènent)
print("✅ Structures KG reconstruites")

Reconstruction des structures KG...
  Cours avec concepts : 708
  Concepts avec prérequis : 19,910
  Concepts menant à d'autres : 17,854
✅ Structures KG reconstruites


In [11]:
import numpy as np
from collections import defaultdict, Counter

# Pré-calculer : pour chaque cours candidat, ses concepts
cand_concepts = {c: concepts_of_course.get(c, set()) for c in candidate_courses_valid}

def kg_cold_start_score(start_course):
    """Score KG pour tous les cours candidats à partir d'un cours de départ."""
    start_concepts = concepts_of_course.get(start_course, set())
    if not start_concepts:
        return None

    # Concepts avancés : ce vers quoi mènent les concepts du cours de départ
    advanced_concepts = set()
    for c in start_concepts:
        advanced_concepts |= concept_leads_to.get(c, set())

    # Score de chaque cours candidat :
    # + s'il couvre les concepts avancés (progression)
    # + s'il partage des concepts (continuité)
    scores = np.zeros(len(candidate_courses_valid))
    for j, cid in enumerate(candidate_courses_valid):
        cc = cand_concepts.get(cid, set())
        if not cc: continue
        # progression : couvre des concepts avancés
        prog = len(cc & advanced_concepts)
        # continuité : partage des concepts de départ
        cont = len(cc & start_concepts)
        scores[j] = prog * 2 + cont  # progression pondérée plus fort
    return scores

# Évaluer le cold-start KG
np.random.seed(42)
sample = np.random.choice(df_test['user_id'].tolist(), 500, replace=False)

ndcg_kg=[]; recall_kg=[]; n_eval=0
for u in sample:
    test_row = df_test[df_test['user_id']==u]
    if len(test_row)==0: continue
    target = test_row['course_id'].iloc[0]
    if target not in cand_valid_index: continue

    history = user_history_courses.get(u, [])
    start_courses = [c for c in history if c != target and concepts_of_course.get(c)]
    if not start_courses: continue

    # Utiliser le 1er cours de départ qui a des concepts
    start_course = start_courses[0]
    scores = kg_cold_start_score(start_course)
    if scores is None or scores.max() == 0: continue

    target_pos = cand_valid_index[target]
    rank = int((scores > scores[target_pos]).sum()) + 1
    if rank <= 10:
        ndcg_kg.append(1.0/np.log2(rank+1)); recall_kg.append(1.0)
    else:
        ndcg_kg.append(0.0); recall_kg.append(0.0)
    n_eval += 1

print(f"=== COLD-START via KG (progression pédagogique) ===")
print(f"  NDCG@10 : {np.mean(ndcg_kg):.4f}")
print(f"  Recall@10 : {np.mean(recall_kg):.4f}")
print(f"  Users évalués : {n_eval}")
print(f"\n  === COMPARAISON DES APPROCHES COLD-START ===")
print(f"  Hasard : 0.0004")
print(f"  Texte (domaines) : 0.0089")
print(f"  Texte (few-shot) : 0.0078")
print(f"  KG (progression) : {np.mean(ndcg_kg):.4f}")
print(f"  Warm (complet) : 0.1136")

=== COLD-START via KG (progression pédagogique) ===
  NDCG@10 : 0.0918
  Recall@10 : 0.2520
  Users évalués : 500

  === COMPARAISON DES APPROCHES COLD-START ===
  Hasard : 0.0004
  Texte (domaines) : 0.0089
  Texte (few-shot) : 0.0078
  KG (progression) : 0.0918
  Warm (complet) : 0.1136


In [12]:
import json, os

out_dir = r'D:\Memoire\LLMKGER\explanations'
os.makedirs(out_dir, exist_ok=True)

l7_results = {
    'cold_start_strategies': {
        'random_baseline': 0.0004,
        'text_domains': 0.0089,
        'text_fewshot': 0.0078,
        'kg_progression': float(np.mean(ndcg_kg)),
        'warm_full_system': 0.1136,
    },
    'kg_progression_detail': {
        'ndcg@10': float(np.mean(ndcg_kg)),
        'recall@10': float(np.mean(recall_kg)),
        'users_evaluated': n_eval,
    },
    'conclusion': 'KG-based cold-start (pedagogical progression via prerequisites) '
                  'vastly outperforms content-based text matching (0.0918 vs 0.0089) '
                  'and approaches warm performance (0.1136), proving the KG captures '
                  'learning progression logic for new users without interaction history.'
}

with open(os.path.join(r'D:\Memoire\LLMKGER\reco_results_final', 'L7_cold_start_results.json'), 'w') as f:
    json.dump(l7_results, f, indent=2, ensure_ascii=False)

print("=== L7 COLD-START — RÉSULTATS FINAUX ===\n")
print("Stratégie                NDCG@10")
print("-"*35)
print(f"Hasard                   0.0004")
print(f"Texte (domaines)         0.0089")
print(f"Texte (few-shot)         0.0078")
print(f"KG (progression)         {np.mean(ndcg_kg):.4f}  ← MEILLEUR")
print(f"Warm (complet)           0.1136")
print(f"\nRecall@10 KG : {np.mean(recall_kg):.4f}")
print(f"\n💾 Sauvegardé : L7_cold_start_results.json")

=== L7 COLD-START — RÉSULTATS FINAUX ===

Stratégie                NDCG@10
-----------------------------------
Hasard                   0.0004
Texte (domaines)         0.0089
Texte (few-shot)         0.0078
KG (progression)         0.0918  ← MEILLEUR
Warm (complet)           0.1136

Recall@10 KG : 0.2520

💾 Sauvegardé : L7_cold_start_results.json


In [3]:
import numpy as np, json, os

# Dossier de sortie pour l'app Flask
APP_DATA = r'D:\Memoire\llmkger_app\model_data'
os.makedirs(APP_DATA, exist_ok=True)

# 1. Sauvegarder les embeddings des cours (déjà calculés dans L7)
np.save(os.path.join(APP_DATA, 'course_embeddings.npy'), course_embeddings.astype(np.float32))
print(f"✅ Embeddings sauvegardés : {course_embeddings.shape}")

# 2. Sauvegarder les métadonnées des cours (nom, domaines) alignées avec les embeddings
course_meta = []
for cid in course_ids:
    info = course_info.get(cid, {})
    course_meta.append({
        'id': cid,
        'name': info.get('name', cid),
        'field': info.get('field', []),
    })
with open(os.path.join(APP_DATA, 'course_meta.json'), 'w', encoding='utf-8') as f:
    json.dump(course_meta, f, ensure_ascii=False)
print(f"✅ Métadonnées sauvegardées : {len(course_meta)} cours")

# 3. Extraire la liste unique des domaines (pour les cases à cocher)
all_fields = set()
for cid in course_ids:
    for fld in course_info.get(cid, {}).get('field', []):
        if fld and fld.strip():
            all_fields.add(fld.strip())
all_fields = sorted(all_fields)
with open(os.path.join(APP_DATA, 'fields.json'), 'w', encoding='utf-8') as f:
    json.dump(all_fields, f, ensure_ascii=False)
print(f"✅ Domaines sauvegardés : {len(all_fields)} domaines")
print(f"\nExemples de domaines : {all_fields[:15]}")

✅ Embeddings sauvegardés : (3781, 384)
✅ Métadonnées sauvegardées : 3781 cours
✅ Domaines sauvegardés : 111 domaines

Exemples de domaines : ['Agricultural and Forestry Economic Management', 'Animal Husbandry', 'Applied Economics', 'Architecture', 'Art', 'Arts', 'Atmospheric science', 'Basic Chinese Medicine', 'Basic Medicine', 'Battle Studies', 'Biology', 'Biomedical Engineering', 'Business Administration', 'Chemical', 'Chemical Engineering and Technology']


In [4]:
import matplotlib.pyplot as plt, numpy as np, os
RESULTS_REC = r'D:\Memoire\LLMKGER\reco_results_final'
strategies = ['Random\nbaseline','Text matching\n(domains)','Text matching\n(1 course)','KG Progression\n(proposed)']
ndcg   = [0.0016, 0.0089, 0.0078, 0.0918]   # random NDCG corrige : 0.0004 -> 0.0016
recall = [0.0036, 0.0180, 0.0180, 0.2520]
x = np.arange(len(strategies)); w = 0.35
fig, ax = plt.subplots(figsize=(10,6))
b1 = ax.bar(x-w/2, ndcg,   w, label='NDCG@10',   color='#2E86AB', edgecolor='black', alpha=0.85)
b2 = ax.bar(x+w/2, recall, w, label='Recall@10', color='#E63946', edgecolor='black', alpha=0.85)
for bb,v in list(zip(b1,ndcg))+list(zip(b2,recall)):
    if v>=0.001: ax.text(bb.get_x()+bb.get_width()/2, bb.get_height()+0.004, f'{v:.4f}', ha='center', va='bottom', fontsize=8.5, fontweight='bold')
for b in (b1[3],b2[3]): b.set_hatch('///'); b.set_edgecolor('#1D3557'); b.set_linewidth(2)
ax.set_ylabel('Score', fontsize=12, fontweight='bold'); ax.set_xticks(x); ax.set_xticklabels(strategies, fontsize=10)
ax.set_ylim(0,0.30); ax.legend(fontsize=10, loc='upper left'); ax.yaxis.grid(True, ls='--', alpha=0.4); ax.set_axisbelow(True)
ax.set_title('Cold-start Performance (single-course protocol)', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.savefig(os.path.join(RESULTS_REC,'coldstart_comparison.png'), dpi=150, bbox_inches='tight'); plt.close()
print("coldstart_comparison.png régénéré")

coldstart_comparison.png régénéré


In [3]:
import json, os

out_dir = r'D:\Memoire\LLMKGER\explanations'
os.makedirs(out_dir, exist_ok=True)

l7_results = {
    'cold_start_strategies': {
        'random_baseline': 0.0016,
        'text_domains': 0.0089,
        'text_fewshot': 0.0078,
        'kg_progression': 0.0918,
    },
    'kg_progression_detail': {
        'ndcg@10': 0.0918,
        'recall@10': 0.2520,
        'users_evaluated': 500,
    },
    'note_on_warm': 'The warm full-ranking system is NOT comparable to the cold-start '
                    'setting (different protocol; the starting and target courses are '
                    'drawn from the same learner history and are topically adjacent by '
                    'construction). It is therefore excluded from this comparison.',
    'conclusion': 'Within the controlled cold-start protocol (single held-out target, '
                  'same candidate set), KG-based pedagogical progression via prerequisites '
                  'clearly outperforms content-based text matching (0.0918 vs 0.0089) and '
                  'the random baseline, indicating that the prerequisite structure captures '
                  'learning-progression signal for learners without interaction history.'
}

with open(os.path.join(r'D:\Memoire\LLMKGER\reco_results_final', 'L7_cold_start_results.json'), 'w') as f:
    json.dump(l7_results, f, indent=2, ensure_ascii=False)

print("=== L7 COLD-START — RÉSULTATS FINAUX ===\n")
print("Stratégie                NDCG@10")
print("-"*35)
print("Hasard                   0.0016")
print("Texte (domaines)         0.0089")
print("Texte (few-shot)         0.0078")
print("KG (progression)         0.0918  <- MEILLEUR (cold-start)")
print("\nRecall@10 KG : 0.2520")
print("\n(Warm non comparable : protocole different, exclu)")
print("\n💾 Sauvegardé : L7_cold_start_results.json")

=== L7 COLD-START — RÉSULTATS FINAUX ===

Stratégie                NDCG@10
-----------------------------------
Hasard                   0.0016
Texte (domaines)         0.0089
Texte (few-shot)         0.0078
KG (progression)         0.0918  <- MEILLEUR (cold-start)

Recall@10 KG : 0.2520

(Warm non comparable : protocole different, exclu)

💾 Sauvegardé : L7_cold_start_results.json
